# 双模型分子打分器

## SMILES GAT (108k) × XYZ+BO GNN (102k) 联合评估候选配合物

```
候选SMILES → SMILES GAT → 7性质(gap/homo/lumo/E_elec/...)  ✅ 主模型
            ↓
            RDKit生成3D构象 → XYZ+BO GNN → dipole  ✅ 偶极矩专项优化
```


In [ ]:
# ============================================================
# Step 1: 导入 + 特征工程
# ============================================================
import numpy as np, pandas as pd, torch, torch.nn as nn, warnings, os
warnings.filterwarnings('ignore')
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ELEMENT_DATA = {
    1:(1,1),5:(2,13),6:(2,14),7:(2,15),8:(2,16),9:(2,17),
    14:(3,14),15:(3,15),16:(3,16),17:(3,17),
    20:(4,2),21:(4,3),22:(4,4),23:(4,5),24:(4,6),
    25:(4,7),26:(4,8),27:(4,9),28:(4,10),29:(4,11),30:(4,12),
    35:(4,17),42:(5,6),44:(5,8),45:(5,9),46:(5,10),
    47:(5,11),48:(5,12),53:(5,17),
    72:(6,4),73:(6,5),74:(6,6),75:(6,7),76:(6,8),
    77:(6,9),78:(6,10),79:(6,11),80:(6,12),
}
BOND_TYPES = [1, 2, 3, 12]
B2I = {b: i for i, b in enumerate(BOND_TYPES)}
N_BT = len(BOND_TYPES)

def atom_features(atom):
    z = atom.GetAtomicNum()
    p, g = ELEMENT_DATA.get(z, (0, 0))
    return [z, atom.GetDegree(), atom.GetFormalCharge(), p, g,
            int(atom.GetIsAromatic()), atom.GetTotalNumHs(), int(z >= 21)]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    N = x.size(0)
    src, dst, battr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = int(bond.GetBondType())
        src.extend([i, j]); dst.extend([j, i])
        bf = [0.0] * N_BT
        if bt in B2I: bf[B2I[bt]] = 1.0
        battr.extend([bf, bf])
    ei = torch.tensor([src, dst], dtype=torch.long) if src else torch.zeros(2,0,dtype=torch.long)
    ea = torch.tensor(battr, dtype=torch.float32) if battr else torch.zeros(0, N_BT)
    if ea.size(1) == N_BT:
        ea = torch.cat([ea, torch.zeros(ea.size(0), 1)], dim=1)
    return {'x': x, 'edge_index': ei, 'edge_attr': ea}


In [ ]:
# ============================================================
# Step 2: 模型1 — SMILES GNN (GraphConv, 和checkpoint匹配)
# ============================================================

class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim, edge_dim=0):
        super().__init__()
        self.msg = nn.Linear(in_dim+edge_dim, out_dim)
        self.self = nn.Linear(in_dim, out_dim)
        self.act = nn.ReLU()
    def forward(self, x, ei, ea=None):
        src, dst = ei[0], ei[1]; N = x.size(0)
        inp = torch.cat([x[src], ea], dim=1) if ea is not None and ea.size(0)>0 else x[src]
        msgs = self.msg(inp)
        out = torch.zeros(N, msgs.size(1), device=x.device)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(msgs), msgs)
        return self.act(out + self.self(x))

class SMILES_GNN(nn.Module):
    def __init__(self, node_dim=8, edge_dim=6, hidden=128, n_layers=4, output_dim=8, dropout=0.15):
        super().__init__()
        self.embed = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GraphConv(hidden,hidden,edge_dim) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, output_dim))
    def forward(self, g):
        x, ei, ea = g['x'], g['edge_index'], g.get('edge_attr', None)
        h = self.embed(x)
        for c in self.convs: h = h + c(h, ei, ea); h = self.drop(h)
        return self.readout(torch.cat([h.mean(dim=0), h.max(dim=0).values]))

model_smiles = SMILES_GNN(node_dim=8, edge_dim=6, hidden=128, output_dim=8).to(device)
ckpt = '../chem_ml5/tmqm_best.pt'
if os.path.exists(ckpt):
    model_smiles.load_state_dict(torch.load(ckpt, map_location=device))
    print(f'SMILES GNN loaded from {ckpt}')
else:
    print(f'Checkpoint not found: {ckpt}')
model_smiles.eval()

In [ ]:
# ============================================================
# Step 3: 模型2 — XYZ+BO GNN (GraphConv, 和checkpoint匹配)
# ============================================================

# GraphConv 已在 Step 2 中定义, 此处复用

class XYZ_GNN(nn.Module):
    def __init__(self, node_dim=9, edge_dim=12, hidden=128, n_layers=4, output_dim=8, dropout=0.15):
        super().__init__()
        self.embed = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GraphConv(hidden,hidden,edge_dim) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, output_dim))
    def forward(self, g):
        x, ei, ea = g['x'], g['edge_index'], g.get('edge_attr', None)
        h = self.embed(x)
        for c in self.convs: h = h + c(h, ei, ea); h = self.drop(h)
        return self.readout(torch.cat([h.mean(dim=0), h.max(dim=0).values]))

model_xyz = XYZ_GNN(node_dim=9, edge_dim=12, hidden=128, output_dim=8).to(device)
ckpt_xyz = '../chem_ml5/xyz_best.pt'
has_xyz = os.path.exists(ckpt_xyz)
if has_xyz:
    model_xyz.load_state_dict(torch.load(ckpt_xyz, map_location=device))
    print(f'XYZ+BO GNN loaded from {ckpt_xyz}')
else:
    print(f'XYZ+BO checkpoint not found ({ckpt_xyz}), using SMILES-only scoring')
model_xyz.eval()


def smiles_to_3d_graph(smiles):
    """SMILES → 3D构象 → XYZ格式的图 (用于XYZ+BO模型)"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    try:
        if AllChem.EmbedMolecule(mol, randomSeed=42, maxAttempts=200) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(mol)
    except: return None

    conf = mol.GetConformer()
    N = mol.GetNumAtoms()

    def pgrp(z):
        tbl = {1:(1,1),5:(2,13),6:(2,14),7:(2,15),8:(2,16),9:(2,17),
               14:(3,14),15:(3,15),16:(3,16),17:(3,17),20:(4,2),21:(4,3),
               22:(4,4),23:(4,5),24:(4,6),25:(4,7),26:(4,8),27:(4,9),
               28:(4,10),29:(4,11),30:(4,12),35:(4,17),42:(5,6),44:(5,8),
               45:(5,9),46:(5,10),47:(5,11),48:(5,12),53:(5,17),
               74:(6,6),75:(6,7),76:(6,8),77:(6,9),78:(6,10),79:(6,11),80:(6,12)}
        return tbl.get(z,(0,0))

    xf = []
    for a in mol.GetAtoms():
        z = a.GetAtomicNum()
        p,g = pgrp(z)
        xf.append([z, p, g, int(z>20), int(z==26), int(z in [27,28,29,24,25])])
    x = torch.tensor(xf, dtype=torch.float32)
    c = torch.tensor([[conf.GetAtomPosition(i).x,
                       conf.GetAtomPosition(i).y,
                       conf.GetAtomPosition(i).z] for i in range(N)], dtype=torch.float32)
    dist = torch.norm(c.unsqueeze(0)-c.unsqueeze(1), dim=2)
    dist.fill_diagonal_(float('inf'))
    k = min(8, N-1)
    vals, idx = torch.topk(dist, k, dim=1, largest=False)
    src, dst, ef = [], [], []
    for i in range(N):
        for j in range(k):
            d = vals[i,j].item()
            if d < 5.0:
                src.append(i); src.append(idx[i,j].item())
                dst.append(idx[i,j].item()); dst.append(i)
                feat = [0.0, d]
                for mu in np.linspace(1.0,5.0,6):
                    feat.append(np.exp(-(d-mu)**2/0.8))
                ef.extend([feat, feat])
    ei = torch.zeros(2,0,dtype=torch.long) if not src else torch.tensor([src,dst],dtype=torch.long)
    ea = torch.zeros(0,8) if not src else torch.tensor(ef,dtype=torch.float32)
    return {'x': x, 'edge_index': ei, 'edge_attr': ea}

print('XYZ+BO model ready')

In [ ]:
# ============================================================
# Step 4: 填入候选 + 双模型打分
# ============================================================

# ↓↓↓ 在这里填入你的候选 SMILES ↓↓↓
candidates = {
    'Fe_dichloro_nitrato_diol_a': 'Cl[Fe]Cl.O[N+](=O)[O-].[Fe].OCCCO',
    'Fe_dichloro_nitrato_diol_b': 'Cl[Fe](Cl)(O[N+](=O)[O-])(OCCCO)[Fe]',
    # 继续添加...
}

targets = ['gap', 'homo', 'lumo', 'dipole', 'metal_charge', 'E_elec', 'E_disp', 'polarizability']

print(f'Scoring {len(candidates)} candidates...\n')

results = []
for name, smi in candidates.items():
    # === SMILES GAT ===
    g1 = smiles_to_graph(smi)
    if g1 is None: continue
    g1 = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in g1.items()}
    with torch.no_grad():
        pred_smiles = model_smiles(g1).cpu().numpy()

    # === XYZ+BO GNN ===
    g2 = None
    if has_xyz:
        g2 = smiles_to_3d_graph(smi)
    if g2 is not None:
        g2 = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in g2.items()}
        with torch.no_grad():
            pred_xyz = model_xyz(g2).cpu().numpy()
    else:
        pred_xyz = pred_smiles  # fallback

    # 融合: SMILES管7项, XYZ管dipole(索引3)
    fused = pred_smiles.copy()
    if g2 is not None:
        fused[3] = pred_xyz[3]  # dipole用XYZ版

    results.append({
        'Name': name,
        'gap_Ha':        fused[0],
        'homo_Ha':       fused[1],
        'lumo_Ha':       fused[2],
        'dipole_D(smiles)': pred_smiles[3],
        'dipole_D(xyz)':   pred_xyz[3],
        'metal_q_e':     fused[4],
        'E_elec_Ha':     fused[5],
        'E_disp_Ha':     fused[6],
        'polariz_a03':   fused[7],
    })

df = pd.DataFrame(results).sort_values('E_elec_Ha')
print('=== Results (sorted by E_elec, lower=more stable) ===')
print(df.to_string(index=False))

best = df.iloc[0]
print(f'\nBest: {best["Name"]}')
print(f'  E_elec = {best["E_elec_Ha"]:.1f} Ha')
print(f'  gap    = {best["gap_Ha"]:.4f} Ha = {best["gap_Ha"]*27.2114:.2f} eV')
print(f'  dipole = {best["dipole_D(smiles)"]:.2f} D (SMILES) / {best["dipole_D(xyz)"]:.2f} D (XYZ)')
print(f'\nUse "dipole_D(xyz)" for dipole comparisons.')


In [ ]:
# 可视化排名
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
names = df['Name'].values
for ax, col, title, color in [
    (axes[0], 'E_elec_Ha', 'E_elec (lower=more stable)', 'steelblue'),
    (axes[1], 'gap_Ha', 'HOMO-LUMO Gap (Ha)', 'darkorange'),
    (axes[2], 'dipole_D(xyz)', 'Dipole from XYZ+BO GNN (Debye)', 'forestgreen'),
]:
    ax.barh(range(len(names)), df[col].values, color=color)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(col)
    ax.set_title(title)
    ax.invert_yaxis()
plt.suptitle('Candidate Scoring (SMILES GAT + XYZ dipole)', fontsize=14)
plt.tight_layout(); plt.show()


## 输出说明

| 列 | 来源模型 | 说明 |
|----|---------|------|
| E_elec_Ha | SMILES GAT | 按此项排序, 越负越稳定 |
| gap_Ha / homo_Ha / lumo_Ha | SMILES GAT | 7性质主模型 |
| dipole_D(smiles) | SMILES GAT | 仅供参考 (R2~0.42) |
| **dipole_D(xyz)** | **XYZ+BO GNN** | **偶极矩优先使用此项** |

> 如果 XYZ+BO checkpoint 不存在, dipole_D(xyz) 退化为 SMILES 版
